In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ------------------------
# Load pre-normalized data
# ------------------------
OUTPUT_DIR = "windows"

try:
    train = pickle.load(open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "rb"))
    companies = pickle.load(open(os.path.join(OUTPUT_DIR, "train_company_list.pkl"), "rb"))
    print("✅ Loaded 'train_windows.pkl' and 'train_company_list.pkl'.")
except FileNotFoundError:
    print(f"❌ ERROR: Could not find .pkl files in '{OUTPUT_DIR}' folder.")
    print("Please run the preprocessing script first.")
    exit()

# ------------------------
# Encode companies
# ------------------------
enc = LabelEncoder().fit(companies)
train = [(X, y, enc.transform([t])[0]) for X, y, t in train]
print("✅ Encoded company tickers to integers.")

# ------------------------
# Dataset and DataLoader
# ------------------------
class WindowDataset(Dataset):
    def __init__(self, data): 
        self.data = data
    def __len__(self): 
        return len(self.data)
    def __getitem__(self, idx):
        X, y, c = self.data[idx]
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        y = 0.0 if not np.isfinite(y) else y

        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(c, dtype=torch.long)
        )

train_loader = DataLoader(WindowDataset(train), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

# ------------------------
# Transformer Model (with normalization)
# ------------------------
class StockTransformer(nn.Module):
    def __init__(self, feature_dim, num_companies):
        super().__init__()
        d_model = 128
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.company_emb = nn.Embedding(num_companies, d_model)

        self.norm = nn.LayerNorm(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x, c):
        x = self.input_proj(x)
        x = x + self.company_emb(c).unsqueeze(1)
        x = self.norm(x)
        x = self.encoder(x)
        return self.head(x[:, -1, :]).squeeze(-1)

# ------------------------
# Initialize model, loss, optimizer
# ------------------------
feature_dim = train[0][0].shape[1]
model = StockTransformer(feature_dim, len(companies)).to(DEVICE)
print(f"✅ Model created with feature_dim={feature_dim}, num_companies={len(companies)}")

opt = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
loss_fn = nn.MSELoss()

# ------------------------
# Training loop
# ------------------------
EPOCHS = 1500
GRAD_CLIP = 0.5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    skipped = 0

    for X, y, c in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        X, y, c = X.to(DEVICE), y.to(DEVICE), c.to(DEVICE)

        opt.zero_grad()
        pred = model(X, c)

        if torch.isnan(pred).any() or torch.isinf(pred).any():
            skipped += 1
            continue

        loss = loss_fn(pred, y)

        if torch.isnan(loss) or torch.isinf(loss):
            skipped += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

        total_loss += loss.item()

    valid_batches = len(train_loader) - skipped
    avg_loss = total_loss / (valid_batches + 1e-8)
    print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.6f} | Skipped: {skipped}")

# ------------------------
# Save model
# ------------------------
torch.save({
    "model_state": model.state_dict(),
    "companies": companies
}, "model.pt")

print("✅ Model and metadata saved as 'model.pt'")

Using device: cuda
✅ Loaded 'train_windows.pkl' and 'train_company_list.pkl'.
✅ Encoded company tickers to integers.
✅ Model created with feature_dim=12, num_companies=10


Epoch 1/1500: 100%|██████████| 125/125 [00:02<00:00, 41.93it/s]


Epoch 1 | Train Loss: 29469.763201 | Skipped: 0


Epoch 2/1500: 100%|██████████| 125/125 [00:02<00:00, 60.27it/s]


Epoch 2 | Train Loss: 28897.063318 | Skipped: 0


...

Epoch 1498 | Train Loss: 27.012024 | Skipped: 0


Epoch 1499/1500: 100%|██████████| 125/125 [00:01<00:00, 65.50it/s]


Epoch 1499 | Train Loss: 26.727949 | Skipped: 0


Epoch 1500/1500: 100%|██████████| 125/125 [00:01<00:00, 70.31it/s]

Epoch 1500 | Train Loss: 26.591733 | Skipped: 0
✅ Model and metadata saved as 'model.pt'


In [8]:
import os, pickle

train_path = "windows/train_windows.pkl"
company_path = "windows/train_company_list.pkl"

if os.path.exists(train_path):
    train = pickle.load(open(train_path, "rb"))
    print(f"✅ train_windows.pkl loaded: {len(train)} samples")
    if len(train) > 0:
        print(f"Example window shape: {train[0][0].shape}, Example ticker: {train[0][2]}")
    else:
        print("❌ No samples found inside train_windows.pkl")
else:
    print("❌ Missing 'train_windows.pkl' file")

if os.path.exists(company_path):
    companies = pickle.load(open(company_path, "rb"))
    print(f"✅ company list loaded: {len(companies)} companies: {companies}")


✅ train_windows.pkl loaded: 7940 samples
Example window shape: (8, 12), Example ticker: XOM_stock_gdelt_final_train
✅ company list loaded: 10 companies: ['XOM_stock_gdelt_final_train', 'NVDA_stock_gdelt_final_train', 'V_stock_gdelt_final_train', 'TSLA_stock_gdelt_final_train', 'GOOG_stock_gdelt_final_train', 'JPM_stock_gdelt_final_train', 'PFE_stock_gdelt_final_train', 'AMZN_stock_gdelt_final_train', 'MSFT_stock_gdelt_final_train', 'AAPL_stock_gdelt_final_train']
